# Customer Support RAG Engine & Automated LLM Evaluation Benchmark

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) system for customer service policy retrieval, paired with an automated **LLM-as-a-Judge** framework to evaluate Faithfulness, Answer Relevance, and Instruction Compliance against curated Ground Truth test cases.

In [ ]:
# Install required libraries
!pip install -q langchain langchain-community langchain-groq chromadb langchain-huggingface sentence-transformers pandas tabulate

In [ ]:
import os
import warnings
import pandas as pd
import getpass
from tabulate import tabulate

# Suppress deprecation and library warnings for clean logging
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 1. Model & API Configuration
We configure Groq Cloud endpoints utilizing **Qwen 2.5 (27B)** for generation and **GPT-OSS (120B)** as the evaluation judge.

In [ ]:
# Configure Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

# Models assigned for generation and evaluation
GENERATOR_MODEL = "qwen/qwen3.6-27b"
JUDGE_MODEL = "openai/gpt-oss-120b"

print(f"--> Generator Model Assigned : {GENERATOR_MODEL}")
print(f"--> Evaluation Judge Model   : {JUDGE_MODEL}")

Enter Groq API Key: ··········
--> Generator Model Assigned : qwen/qwen3.6-27b
--> Evaluation Judge Model   : openai/gpt-oss-120b


## 2. Knowledge Base Ingestion
We define the official Standard Operating Procedures (SOP) dataset for digital payment refunds, KYC verification, and transaction incident management.

In [ ]:
knowledge_base_text = """
# Digital Pay & E-Commerce Customer Support Operating Procedures

1. Refund Policies & Timelines:
- Refund requests must be submitted within 7 business days of item delivery or transaction processing.
- Transactions made via Virtual Account, Instant Bank Transfer, and QR Code will be refunded within 1 business day (24 hours) directly to the user's in-app wallet balance.
- Credit Card payments require 5 to 14 business days to reflect on the account, depending on the issuing bank.
- Initial transaction and payment processing fees are non-refundable if the cancellation is initiated by the buyer.

2. Account Security & Identity Verification (KYC):
- Users must complete Tier-2 Identity Verification (Government ID upload and biometric selfie scan) for single transactions exceeding $1,000.
- User accounts will be automatically suspended temporarily after 5 consecutive failed PIN or password attempts.
- Account unlocking requires identity verification via a scheduled live video session with an authorized Customer Support specialist.

3. Handling Pending & Failed Transactions:
- If a transaction remains in 'Pending' status for more than 30 minutes, the automated system queries the payment gateway partner.
- If funds are deducted from the user balance but the order fails, an automated system auto-reversal will credit the full amount within 3 business days.
"""

with open("knowledge_base.txt", "w", encoding="utf-8") as f:
    f.write(knowledge_base_text.strip())

print("Knowledge base created successfully: 'knowledge_base.txt'")

Knowledge base created successfully: 'knowledge_base.txt'


## 3. Text Chunking & Dense Vector Indexing (ChromaDB)
The document is partitioned into semantic chunks using `RecursiveCharacterTextSplitter` and embedded into a local Chroma vector database using `all-MiniLM-L6-v2`.

In [ ]:
# Load knowledge base document
loader = TextLoader("knowledge_base.txt", encoding="utf-8")
docs = loader.load()

# Split text into overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(docs)

# Initialize local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Construct vector database
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="customer_support_kb"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Successfully indexed {len(splits)} chunks into ChromaDB.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully indexed 6 chunks into ChromaDB.


## 4. RAG Pipeline Construction & Anti-Hallucination Guardrails
We assemble the LangChain pipeline with strict zero-shot system constraints to prevent model hallucinations on queries outside the retrieved context.

In [ ]:
llm = ChatGroq(model_name=GENERATOR_MODEL, temperature=0.0)

rag_template = """You are an official Customer Support AI assistant.
Answer the user's question strictly based ONLY on the provided context below.
If the answer cannot be found in the context, explicitly state: "This information is not available in our official knowledge base."
Do NOT assume, infer, or hallucinate external facts.

Context:
{context}

Question: {question}

Helpful & Accurate Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain initialized successfully.")

RAG chain initialized successfully.


## 5. Benchmark Test Suite & Ground Truth Formulation
We curate a structured test suite covering Direct Fact Retrieval, Constraint Verification, and Out-of-Domain / Negative Testing.

In [ ]:
test_cases = [
    {
        "id": "TC-01",
        "category": "Direct Fact Retrieval",
        "question": "How long does a refund take if paid via QR Code or Virtual Account?",
        "ground_truth": "Within 1 business day (24 hours) directly to the in-app wallet balance."
    },
    {
        "id": "TC-02",
        "category": "Direct Fact Retrieval",
        "question": "How many failed PIN attempts will cause an account to be suspended?",
        "ground_truth": "5 consecutive failed PIN or password attempts."
    },
    {
        "id": "TC-03",
        "category": "Out-of-Domain / Hallucination Test",
        "question": "How can I request a refund for an international flight ticket?",
        "ground_truth": "This information is not available in our official knowledge base."
    },
    {
        "id": "TC-04",
        "category": "Constraint Verification",
        "question": "Are initial processing fees refunded if a customer cancels the order?",
        "ground_truth": "Initial transaction and processing fees are non-refundable if initiated by the buyer."
    }
]

print(f"Defined {len(test_cases)} evaluation test cases.")

Defined 4 evaluation test cases.


## 6. Automated Evaluation Framework (LLM-as-a-Judge)
An objective, high-parameter LLM judges the candidate bot answers across three core metrics:
- **Faithfulness**: Detection of fabricated or hallucinated claims.
- **Answer Relevance**: Directness and conciseness relative to the question.
- **Instruction Compliance**: Strict adherence to boundary constraints.

In [ ]:
eval_judge_llm = ChatGroq(model_name=JUDGE_MODEL, temperature=0.0)

judge_template = """You are an expert AI Quality Assurance and Data Evaluation Specialist.
Evaluate the candidate bot answer against the provided Context and Ground Truth reference.

Context: {context}
User Question: {question}
Ground Truth: {ground_truth}
Bot Answer: {bot_answer}

Provide your evaluation strictly following this format:
- Faithfulness: [Score 1 if purely grounded in context, 0 if hallucinated]
- Answer Relevance: [Score 1 if directly answers question, 0 if irrelevant]
- Instruction Compliance: [Score 1 if adhered to guardrail/out-of-domain rules, 0 if violated]
- Rationale: [One concise sentence explaining the score]"""

judge_prompt = ChatPromptTemplate.from_template(judge_template)
judge_chain = judge_prompt | eval_judge_llm | StrOutputParser()

print("Evaluation Judge pipeline configured.")

Evaluation Judge pipeline configured.


## 7. Execution, Metrics Summary, & Dataset Export
Execute the benchmark over all test cases, log retrieved chunks, record model responses, run evaluation scoring, and export the structured results to CSV.

In [ ]:
results = []
print("Starting automated benchmark evaluation...\n")

for tc in test_cases:
    print(f"Executing: {tc['id']} - {tc['category']}")

    # 1. Retrieve context
    retrieved_chunks = retriever.invoke(tc["question"])
    context_text = format_docs(retrieved_chunks)

    # 2. Generate RAG output
    bot_response = rag_chain.invoke(tc["question"])

    # 3. Evaluate output with Judge
    feedback = judge_chain.invoke({
        "context": context_text,
        "question": tc["question"],
        "ground_truth": tc["ground_truth"],
        "bot_answer": bot_response
    })

    results.append({
        "Test ID": tc["id"],
        "Category": tc["category"],
        "Question": tc["question"],
        "Ground Truth": tc["ground_truth"],
        "Bot Output": bot_response,
        "Judge Evaluation": feedback
    })

# Export evaluation logs to CSV
df_results = pd.DataFrame(results)
df_results.to_csv("rag_evaluation_results.csv", index=False)

print("\n" + "="*80)
print("BENCHMARK EXECUTION SUMMARY")
print("="*80)
print(tabulate(df_results[["Test ID", "Category", "Bot Output"]], headers="keys", tablefmt="grid"))
print("\nEvaluation successfully exported to 'rag_evaluation_results.csv'.")

Starting automated benchmark evaluation...

Executing: TC-01 - Direct Fact Retrieval
Executing: TC-02 - Direct Fact Retrieval
Executing: TC-03 - Out-of-Domain / Hallucination Test
Executing: TC-04 - Constraint Verification

BENCHMARK EXECUTION SUMMARY
+----+-----------+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|    | Test ID   | Category                           | Bot Output                                                                                    

In [ ]:
# Inspect detailed Judge scoring feedback for each test case
for i, row in df_results.iterrows():
    print(f"\n==================== {row['Test ID']} : {row['Category']} ====================")
    print(f"Prompt        : {row['Question']}")
    print(f"Ground Truth  : {row['Ground Truth']}")
    print(f"\nJudge Feedback:\n{row['Judge Evaluation']}")


==================== TC-01 : Direct Fact Retrieval ====================
Prompt        : How long does a refund take if paid via QR Code or Virtual Account?
Ground Truth  : Within 1 business day (24 hours) directly to the in-app wallet balance.

Judge Feedback:
- Faithfulness: 1
- Answer Relevance: 1
- Instruction Compliance: 1
- Rationale: The response directly mirrors the context, correctly states the 1‑business‑day refund timeline for QR Code and Virtual Account payments, and contains no extraneous or prohibited content.

==================== TC-02 : Direct Fact Retrieval ====================
Prompt        : How many failed PIN attempts will cause an account to be suspended?
Ground Truth  : 5 consecutive failed PIN or password attempts.

Judge Feedback:
- Faithfulness: 1
- Answer Relevance: 1
- Instruction Compliance: 1
- Rationale: The response directly quotes the context, correctly states that 5 consecutive failed PIN (or password) attempts trigger a temporary suspension, and foll